In [41]:
import pandas as pd

# apri il file scurve_fit_results.csv e mostra le prime righe
verify_df = pd.read_csv("scurve_tdac0_fit_results.csv")

# calcola e mostra la media della colonna 'mu', ovvero la soglia media della matrice
mean_mu = verify_df['mu'].mean()
print(f"Mean mu: {mean_mu}")


def compute_tdac_corrections(df, target_mu=None, tdac_step_size=0.50):
    """
    Per ogni pixel calcola:
      - error_mu: valore assoluto della differenza tra target_mu e mu del pixel
      - tup: 1 se il mu del pixel > target_mu (serve a diminuire il mu), altrimenti 0
      - tdac_steps: numero intero di step TDAC necessari (troncamento)
    Restituisce una lista di tuple: (row, col, tup, tdac_steps)
    """
    if target_mu is None:
        target_mu = df['mu'].mean()
    results = []
    
    # Assicurati che 'row' e 'col' siano trattati come interi
    # df['row'] = df['row'].astype(int) 
    # df['col'] = df['col'].astype(int)
    
    for _, pix in df.iterrows():

        mu = pix['mu']
        error_mu = abs(target_mu - mu)
        
        # tup = 1 se il mu del pixel > target_mu (diminuire mu), altrimenti 0 (aumentare mu)
        tup = 0 if mu > target_mu else 1
        
        tdac_steps = (error_mu / 30.0)*31
        #arrotonda tdac_steps all'intero più vicino
        tdac_steps = int(round(tdac_steps))
        
        # Inserisci i valori in una tupla
        results.append((int(pix['row']), int(pix['col']), tup, tdac_steps)) 
        
    return results

# uso: calcola le correzioni usando la mean_mu già presente
tdac_corrections = compute_tdac_corrections(verify_df, target_mu=mean_mu)
print(tdac_corrections)



Mean mu: 55.189296875
[(0, 0, 1, 7), (0, 1, 0, 10), (0, 2, 1, 3), (0, 3, 0, 8), (0, 4, 1, 7), (0, 5, 1, 6), (0, 6, 1, 14), (0, 7, 0, 11), (1, 0, 0, 7), (1, 1, 1, 4), (1, 2, 0, 10), (1, 3, 1, 7), (1, 4, 1, 8), (1, 5, 0, 10), (1, 6, 1, 7), (1, 7, 1, 6), (2, 0, 1, 8), (2, 1, 0, 2), (2, 2, 0, 7), (2, 3, 0, 11), (2, 4, 0, 10), (2, 5, 0, 10), (2, 6, 1, 5), (2, 7, 0, 13), (3, 0, 0, 2), (3, 1, 1, 2), (3, 2, 0, 0), (3, 3, 0, 0), (3, 4, 1, 10), (3, 5, 0, 11), (3, 6, 0, 12), (3, 7, 0, 8), (4, 0, 0, 0), (4, 1, 1, 11), (4, 2, 1, 6), (4, 3, 1, 1), (4, 4, 1, 13), (4, 5, 1, 4), (4, 6, 1, 5), (4, 7, 0, 6), (5, 0, 1, 4), (5, 1, 0, 15), (5, 2, 0, 4), (5, 3, 0, 12), (5, 4, 1, 5), (5, 5, 0, 6), (5, 6, 0, 2), (5, 7, 0, 11), (6, 0, 1, 3), (6, 1, 0, 1), (6, 2, 1, 4), (6, 3, 1, 6), (6, 4, 1, 5), (6, 5, 0, 6), (6, 6, 1, 7), (6, 7, 1, 3), (7, 0, 1, 12), (7, 1, 1, 7), (7, 2, 0, 3), (7, 3, 1, 1), (7, 4, 1, 7), (7, 5, 0, 7), (7, 6, 1, 3), (7, 7, 0, 5), (8, 0, 1, 6), (8, 1, 1, 2), (8, 2, 0, 3), (8, 3, 0, 5), (8, 4, 

In [42]:
# crea un file tsv con le colonne: pixel_index, t_up_tuner, tdac_tuner
with open("tdac_tuning_results.tsv", "w") as f:
    f.write("row\tcol\tt_up_tuner\ttdac_tuner\n")
    for index, (row, col, t_up_tuner, tdac_tuner) in enumerate(tdac_corrections):
        f.write(f"{row}\t{col}\t{t_up_tuner}\t{tdac_tuner}\n")